# LLM-as-a-Judge Inference

This notebook performs large-scale inference for preference generation.
Given constructed judge prompts, we generate model outputs using either:

1. Repeated sampling,

2. Hint Sampling

The goal is to collect diverse candidate outputs that will later be evaluated and converted into DPO training pairs.

This stage bridges dataset construction and preference optimization.


### Loading Prompt Records

We load previously constructed judge prompts from disk.
Each record contains:

- A structured evaluation prompt
- Metadata about the original example
- A ground-truth chosen label 

This ensures compatibility with downstream DPO training.

In [ ]:
import os; os.environ.setdefault('HF_HUB_DISABLE_XET', '1')  # avoid full-repo downloads
import contextlib
import os
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils.inference_helpers import (
    build_prompt_records,
    load_arrow_records,
    load_disk_records,
    run_batched_inference,
    run_best_of_n,
)


# Configuration
MODE = "best_of_n"  # options: "best_of_n" | "batched"
MODEL_ID = "Qwen/Qwen2-7B-Instruct"

# Resolve Repository Root
try:
    REPO_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    REPO_ROOT = Path.cwd()

# Dataset Folder (must match Notebook 01)
DATA_FOLDER_NAME = "data_sky"  # <-- change if needed

# Expected processed dataset directory
PROCESSED_DATASET_DIR = REPO_ROOT / f"output_{DATA_FOLDER_NAME}"

if not PROCESSED_DATASET_DIR.exists():
    raise FileNotFoundError(f"{PROCESSED_DATASET_DIR} not found. Please run 01_dataset_construction.ipynb first.")

# HuggingFace Arrow Dataset Path (auto-detect)
train_dir = PROCESSED_DATASET_DIR / "train"

arrow_files = list(train_dir.glob("*.arrow"))

if not arrow_files:
    raise FileNotFoundError(
        f"No .arrow files found in {train_dir}. Ensure dataset construction completed successfully."
    )

ARROW_DATASET_PATH = arrow_files[0]

print(f"Using Arrow dataset: {ARROW_DATASET_PATH}")


# Output Directories (auto-based on dataset)
OUT_DIR = REPO_ROOT / f"outputs_{DATA_FOLDER_NAME}"
CKPT_DIR = OUT_DIR / "checkpoints"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Outputs will be saved to: {OUT_DIR}")

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

# Model
torch.backends.cuda.matmul.allow_tf32 = True
with contextlib.suppress(Exception):
    torch.set_float32_matmul_precision("high")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",  # Choose "flash_attention" or "sdpa" based on your GPU capabilities
).eval()

print("Model loaded:", MODEL_ID)
print("Attention backend:", model.config._attn_implementation)

### Repeated Sampling

We generate multiple stochastic completions per prompt using nucleus sampling.

Repeated sampling increases output diversity and approximates exploration used in RLHF-style pipelines.
Multiple candidate generations enable more robust preference construction during later ranking or filtering.

Sampling parameters:

- temperature = 1.2
- top_p = 0.95

This balances exploration and coherence, avoiding degenerate outputs while preserving variability.

In [ ]:
if MODE == "best_of_n":
    records = load_disk_records(train_dir, limit=20)  # Change to 200 for larger runs

    run_best_of_n(
        records=records,
        model=model,
        tokenizer=tokenizer,
        output_path=os.path.join(OUT_DIR, "best_of_n.jsonl"),
        checkpoint_dir=os.path.join(CKPT_DIR, "best_of_n"),
        task_name="best_of_n",
        n=8,
        checkpoint_every=5,
        max_new_tokens=512,
    )

    print("Best-of-N inference completed")

### Hint Sampling

In this mode, we perform deterministic batched generation using hint-conditioned prompts.
Each prompt explicitly specifies which answer is known to be better ("Known answer X is better") and asks the model to explain why.

We construct two variants:

guide — original ordering
guide_reverse — reversed ordering to control for positional bias

This setup allows us to:

Evaluate whether the model can justify a given preference
Test robustness to answer ordering

In [ ]:
if MODE == "batched":
    templates = {
        "guide": [
            "As an evaluation expert, given a question and its two possible answers, "
            "please analyze the performance of both in terms of coherence, accuracy, "
            "coverage, and the overall quality defined above.\nQuestion:",
            "\nAnswer 1:",
            "\nAnswer 2:",
            "\nKnown answer ",
            " is better, please explain the reason:",
        ],
        "guide_reverse": [
            "As an evaluation expert, given a question and its two possible answers, "
            "please analyze the performance of both in terms of coherence, accuracy, "
            "coverage, and the overall quality defined above.\nQuestion:",
            "\nAnswer 1:",
            "\nAnswer 2:",
            "\nKnown answer ",
            " is better, please explain the reason:",
        ],
    }

    raw = load_arrow_records(str(ARROW_DATASET_PATH), limit=20)  # Change to 200 for larger runs

    batched_outputs = {}

    for key in ["guide", "guide_reverse"]:
        print(f"\n▶ Running batched inference for: {key}")

        records = build_prompt_records(
            dataset=raw,
            templates=templates,
            template_key=key,
            reverse=(key == "guide_reverse"),
        )

        batched_outputs[key] = run_batched_inference(
            records=records,
            model=model,
            tokenizer=tokenizer,
            batch_size=4,
            max_new_tokens=512,
        )

        # 🔍 Inspect a few samples (CORRECT)
        for r in batched_outputs[key][:5]:
            print(f"[{key}] {r['prompt_idx']} → {r['output'][:120]}")

In [ ]:
import jsonlines


if MODE == "batched":
    for key, results in batched_outputs.items():
        out_path = os.path.join(OUT_DIR, f"{key}.jsonl")

        with jsonlines.open(out_path, "w") as writer:
            for r in results:
                writer.write(r)

        print(f"💾 Saved → {out_path}")